# ActiveDROPS plotting (streamlined)

Lean notebook to load the aggregated motor dataset, apply optional time-window filtering, and generate the core plots with configurable font sizes from a master function.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [2]:
# --- paths & dataset ---
DATA_CANDIDATES = [
    Path.home() / "Downloads" / "motor_dataset.csv",
    Path("../../../Thomson Lab Dropbox/David Larios/activedrops/main/all/motor_dataset.csv"),
    Path.cwd() / "motor_dataset.csv",
]
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("motor_dataset.csv not found in candidates; update DATA_CANDIDATES")

print(f"Using dataset: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH)

df = df_raw.copy()

# Standardize column names we rely on
rename_map = {
    "power [W]_mean": "power_W",
    "work [J]_mean": "work_J",
    "Protein Concentration_nM": "protein_conc_nM",
    "Translation Rate [nM/s]": "translation_rate_nMs",
    "Translation Rate aa_s": "translation_rate_aa_s",
    "DNA nM": "dna_nM",
}
df.rename(columns=rename_map, inplace=True)

num_cols = [
    "dna_nM", "protein_conc_nM", "translation_rate_nMs", "translation_rate_aa_s",
    "power_W", "work_J", "time (s)", "Time_min", "Time_h",
    "velocity magnitude [m/s]_mean", "correlation length [m]_mean",
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Convenience derived columns
if "velocity magnitude [m/s]_mean" in df.columns:
    df["velocity_nm_s"] = df["velocity magnitude [m/s]_mean"] * 1e9
if "correlation length [m]_mean" in df.columns:
    df["corr_len_um"] = df["correlation length [m]_mean"] * 1e6

print(df.head())


Using dataset: /Users/dalarios/Downloads/motor_dataset.csv
     condition subcondition  time (s)  Time_min    Time_h  Mean Intensity  \
0  K401_1p25nM         Rep1       0.0       0.0  0.000000      179.914233   
1  K401_1p25nM         Rep1     600.0      10.0  0.166667      181.127909   
2  K401_1p25nM         Rep1    1200.0      20.0  0.333333      181.868650   
3  K401_1p25nM         Rep1    1800.0      30.0  0.500000      182.932426   
4  K401_1p25nM         Rep1    2400.0      40.0  0.666667      184.062858   

   Protein Concentration_ng_ul  protein_conc_nM  Unnamed: 0  frame  ...  \
0                     0.000000         0.000000         NaN    NaN  ...   
1                     0.051565         1.909813         NaN    NaN  ...   
2                     0.083037         3.075428         NaN    NaN  ...   
3                     0.128233         4.749361         NaN    NaN  ...   
4                     0.176261         6.528183         NaN    NaN  ...   

   time (h)  protein  dna_n

In [3]:
def apply_time_filters(df, rules, time_col="time (s)"):
    """
    Mask velocity/power-like fields outside per-(protein, dna) time windows.
    rules: list of dicts with keys protein, dna, t_min, t_max (in seconds).
    Returns a copy with out-of-window values set to NaN for velocity/power/etc.
    """
    velocity_cols = [
        "velocity magnitude [m/s]_mean", "power_W", "vorticity [1/s]_mean",
        "divergence [1/s]_mean", "strain [1/s]_mean", "shear [1/s]_mean",
        "work_J", "correlation length [m]_mean", "corr_len_um",
    ]
    out = df.copy()
    for rule in rules:
        prot = rule["protein"]
        dna = str(rule["dna"])
        t_min = rule.get("t_min", -np.inf)
        t_max = rule.get("t_max", np.inf)
        msk = (out.get("protein") == prot) & (out.get("dna_nM") == dna)
        if time_col in out:
            msk &= (out[time_col] < t_min) | (out[time_col] > t_max)
        for col in velocity_cols:
            if col in out.columns:
                out.loc[msk, col] = np.nan
    return out

# Example filter set (seconds)
default_rules = [
    {"protein": "A", "dna": 160, "t_min": 300, "t_max": 32*60*60},
    {"protein": "H", "dna": 80, "t_min": 120, "t_max": 60*60},
]

# Commented out by default
# df = apply_time_filters(df, default_rules)



In [4]:
def set_font_sizes(font_sizes=None):
    base = {"title": 22, "label": 20, "tick": 16, "legend": 16, "text": 14}
    if font_sizes:
        base.update(font_sizes)
    plt.rcParams.update({
        "axes.titlesize": base["title"],
        "axes.labelsize": base["label"],
        "xtick.labelsize": base["tick"],
        "ytick.labelsize": base["tick"],
        "legend.fontsize": base["legend"],
    })
    return base

def smooth_series(y, sigma):
    return gaussian_filter1d(np.asarray(y, dtype=float), sigma=sigma)



In [5]:
def _color_map(proteins):
    cmap = plt.cm.tab10(np.linspace(0, 1, max(len(proteins), 3)))
    return {p: cmap[i % len(cmap)] for i, p in enumerate(proteins)}


def plot_protein_conc_vs_time(ax, df, proteins, dna_vals, time_col, sigma=6, label_dna=False):
    colors = _color_map(proteins)
    for p in proteins:
        for i, dna in enumerate(sorted(dna_vals, key=lambda x: float(x), reverse=True)):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            sub = sub.dropna(subset=["protein_conc_nM", time_col])
            if sub.empty:
                continue
            sm = smooth_series(sub["protein_conc_nM"], sigma)
            alpha = 1 - 0.08 * i
            lw = 3 - 0.3 * i
            ax.plot(sub[time_col], sm, color=colors[p], alpha=alpha, lw=lw, label=p if i == 0 else None)
            if label_dna:
                idx = np.argmax(sm)
                ax.text(sub[time_col].iloc[idx], sm[idx], f"{dna}nM", fontsize=12)
    ax.set_xlabel(f"Time ({'h' if 'h' in time_col else 'min'})")
    ax.set_ylabel("Motor (nM)")
    ax.legend()


def plot_velocity_vs_time(ax, df, proteins, dna_vals, time_col, sigma=5, label_dna=False):
    colors = _color_map(proteins)
    for p in proteins:
        for i, dna in enumerate(sorted(dna_vals, key=lambda x: float(x), reverse=True)):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            sub = sub.dropna(subset=["velocity_nm_s", time_col])
            if sub.empty:
                continue
            sm = smooth_series(sub["velocity_nm_s"], sigma)
            alpha = 1 - 0.08 * i
            lw = 3 - 0.3 * i
            ax.plot(sub[time_col], sm, color=colors[p], alpha=alpha, lw=lw, label=p if i == 0 else None)
            if label_dna:
                idx = np.argmax(sm)
                ax.text(sub[time_col].iloc[idx], sm[idx], f"{dna}nM", fontsize=12)
    ax.set_xlabel(f"Time ({'h' if 'h' in time_col else 'min'})")
    ax.set_ylabel("Velocity (nm/s)")
    ax.legend()


def plot_power_vs_time(ax, df, proteins, dna_vals, time_col, sigma=5, label_dna=False):
    colors = _color_map(proteins)
    for p in proteins:
        for i, dna in enumerate(sorted(dna_vals, key=lambda x: float(x), reverse=True)):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            sub = sub.dropna(subset=["power_W", time_col])
            if sub.empty:
                continue
            sm = smooth_series(sub["power_W"], sigma)
            alpha = 1 - 0.08 * i
            lw = 3 - 0.3 * i
            ax.plot(sub[time_col], sm, color=colors[p], alpha=alpha, lw=lw, label=p if i == 0 else None)
            if label_dna:
                idx = np.argmax(sm)
                ax.text(sub[time_col].iloc[idx], sm[idx], f"{dna}nM", fontsize=12)
    ax.set_xlabel(f"Time ({'h' if 'h' in time_col else 'min'})")
    ax.set_ylabel("Power (W)")
    ax.set_yscale('log')
    ax.legend()


def plot_corrlen_vs_time(ax, df, proteins, dna_vals, time_col, sigma=3, label_dna=False):
    colors = _color_map(proteins)
    for p in proteins:
        for i, dna in enumerate(sorted(dna_vals, key=lambda x: float(x), reverse=True)):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            sub = sub.dropna(subset=["corr_len_um", time_col])
            if sub.empty:
                continue
            sm = smooth_series(sub["corr_len_um"], sigma)
            alpha = 1 - 0.08 * i
            lw = 3 - 0.3 * i
            ax.plot(sub[time_col], sm, color=colors[p], alpha=alpha, lw=lw, label=p if i == 0 else None)
            if label_dna:
                idx = np.argmax(sm)
                ax.text(sub[time_col].iloc[idx], sm[idx], f"{dna}nM", fontsize=12)
    ax.set_xlabel(f"Time ({'h' if 'h' in time_col else 'min'})")
    ax.set_ylabel("Correlation length (µm)")
    ax.legend()


def plot_translation_rate(ax, df, proteins, dna_vals, time_col, sigma=3, label_dna=False):
    colors = _color_map(proteins)
    for p in proteins:
        for i, dna in enumerate(sorted(dna_vals, key=lambda x: float(x), reverse=True)):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            sub = sub.dropna(subset=["protein_conc_nM", time_col])
            if sub.empty:
                continue
            conc_sm = smooth_series(sub["protein_conc_nM"], sigma)
            rate = np.gradient(conc_sm, sub[time_col]) / (3600 if 'h' in time_col else 60)
            rate_sm = smooth_series(rate, sigma)
            alpha = 1 - 0.08 * i
            lw = 3 - 0.3 * i
            ax.plot(sub[time_col], rate_sm, color=colors[p], alpha=alpha, lw=lw, label=p if i == 0 else None)
            if label_dna:
                idx = np.argmax(rate_sm)
                ax.text(sub[time_col].iloc[idx], rate_sm[idx], f"{dna}nM", fontsize=12)
    ax.set_xlabel(f"Time ({'h' if 'h' in time_col else 'min'})")
    ax.set_ylabel("Translation rate (nM/s)")
    ax.legend()


def plot_max_velocity_vs_dna(ax, df, proteins, dna_vals):
    colors = _color_map(proteins)
    for p in proteins:
        xs, ys = [], []
        for dna in sorted(dna_vals, key=lambda x: float(x), reverse=True):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            if "velocity_nm_s" not in sub or sub["velocity_nm_s"].dropna().empty:
                continue
            xs.append(float(dna))
            ys.append(sub["velocity_nm_s"].max())
        if not xs:
            continue
        ax.plot(xs, ys, marker='o', color=colors[p], label=p)
    ax.set_xscale('log')
    ax.set_xlabel("DNA (nM)")
    ax.set_ylabel("Max velocity (nm/s)")
    ax.legend()


def plot_total_work_vs_dna(ax, df, proteins, dna_vals):
    colors = _color_map(proteins)
    for p in proteins:
        xs, ys = [], []
        for dna in sorted(dna_vals, key=lambda x: float(x), reverse=True):
            sub = df[(df["protein"] == p) & (df["dna_nM"].astype(str) == str(dna))]
            if "work_J" not in sub or sub["work_J"].dropna().empty:
                continue
            xs.append(float(dna))
            ys.append(sub["work_J"].dropna().iloc[-1])
        if not xs:
            continue
        ax.plot(xs, ys, marker='o', color=colors[p], label=p)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("DNA (nM)")
    ax.set_ylabel("Total work (J)")
    ax.legend()

PLOT_FUNCS = {
    "protein_vs_time": plot_protein_conc_vs_time,
    "velocity_vs_time": plot_velocity_vs_time,
    "power_vs_time": plot_power_vs_time,
    "corrlen_vs_time": plot_corrlen_vs_time,
    "translation_rate": plot_translation_rate,
    "max_velocity_vs_dna": plot_max_velocity_vs_dna,
    "total_work_vs_dna": plot_total_work_vs_dna,
}



In [6]:
def generate_plots(
    df,
    proteins,
    dna_vals,
    plots,
    output_dir,
    time_unit="h",
    font_sizes=None,
    smooth_sigma=5,
    label_dna=False,
    save=True,
    fig_size=(8, 6),
):
    """
    Master plotter. Choose plots by key from PLOT_FUNCS, set font sizes, and save SVGs.
    time_unit: 'h' or 'min' selects the time column if present.
    font_sizes: dict with title/label/tick/legend/text entries.
    smooth_sigma: passed to gaussian filter for time series plots.
    """
    set_font_sizes(font_sizes)
    time_col = "Time_h" if time_unit == "h" and "Time_h" in df.columns else "Time_min"
    if time_col not in df.columns:
        time_col = "time (s)"
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for plot_key in plots:
        func = PLOT_FUNCS.get(plot_key)
        if func is None:
            print(f"Skipping unknown plot key: {plot_key}")
            continue
        fig, ax = plt.subplots(figsize=fig_size)
        if plot_key in {"protein_vs_time", "velocity_vs_time", "power_vs_time", "corrlen_vs_time", "translation_rate"}:
            func(ax, df, proteins, dna_vals, time_col=time_col, sigma=smooth_sigma, label_dna=label_dna)
        else:
            func(ax, df, proteins, dna_vals)
        ax.set_title(plot_key.replace('_', ' ').title())
        plt.tight_layout()
        if save:
            fname = out_dir / f"{plot_key}_{'-'.join(proteins)}_{'-'.join(map(str, dna_vals))}.svg"
            plt.savefig(fname)
        plt.show()



In [7]:
# --- Example usage ---
proteins = ["ThTr", "Kif5", "A"]
dna_vals = [160, 80, 40]
plots = [
    "protein_vs_time",
    "velocity_vs_time",
    "power_vs_time",
    "corrlen_vs_time",
    "translation_rate",
    "max_velocity_vs_dna",
    "total_work_vs_dna",
]
font_sizes = {"title": 20, "label": 18, "tick": 14, "legend": 14}

# Uncomment to run
# generate_plots(
#     df,
#     proteins=proteins,
#     dna_vals=dna_vals,
#     plots=plots,
#     output_dir=Path.cwd() / "streamlined_plots",
#     time_unit="h",
#     font_sizes=font_sizes,
#     smooth_sigma=5,
#     label_dna=False,
#     save=True,
# )
